# Batch detector evaluation — frame review

Ingests the saved offline-batch output (`run_batch_offline_eval.py`) + SigMF ground truth
and renders, for one frame, an **(N+1)-panel** comparison:

* **Panel 0** — spectrogram + ground-truth boxes (filled-box GT mask).
* **Panels 1..N** — the same spectrogram with each detector's mask overlaid.

Spectrogram is reconstructed from the source SigMF (faithful to the binary's FFT grid;
verified to match the saved tensor to ~1e-5 dB), so it works even on masks-only
(`--no-tensors`) sweeps.

Helpers live in `eval_viz.py` + `mask_eval_metrics.py`. This notebook is just a thin driver.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import eval_viz as v
import mask_eval_metrics as mem

# --- parameters ---
# BATCH_ROOT may be ANY of: a batch root (<root>/<detector>/<file_stem>/),
# a detector dir (<root>/<detector>/), or a single run dir (.../<detector>/<file_stem>/).
# resolve_layout() figures out which and normalizes it.
BATCH_ROOT = Path('/tmp/usrp_spectrograms/offline_eval')   # detector dirs live directly under here
CAPTURE_DIRS = [Path('/home/bqn82/captures'),
                Path('../../generated_inputs').resolve()]   # for spectrogram reconstruction
FILE_STEM = 'attenuation_dB_25'   # used to disambiguate when several files are present
DETECTORS = None                  # None = auto-detect all detectors present for this file
FRAME = 100                       # pick a frame with signals (early frames may be empty)

layout = v.resolve_layout(BATCH_ROOT, FILE_STEM)
print('resolved layout:', layout)

## What is available?

In [ ]:
file_stem = layout['file_stem']
detectors_present = layout['detectors']
print('file_stem:', file_stem, '| detectors:', detectors_present)
if detectors_present:
    frames = v.available_frames(layout['batch_root'], detectors_present[0], file_stem)
    print(f'{len(frames)} frames available, e.g.', frames[:10])

## Render the comparison panels for one frame

In [ ]:
bundle = v.load_frame_bundle_smart(BATCH_ROOT, FRAME, file_stem=FILE_STEM,
                                   detectors=DETECTORS, capture_dirs=CAPTURE_DIRS)
print('grid', bundle.fft_rows, 'x', bundle.fft_cols,
      '| span', bundle.span_hz/1e6, 'MHz',
      '| detectors', list(bundle.detector_masks),
      '| GT regions', len(bundle.gt_items))
fig = v.plot_frame_panels(bundle)
fig

## Review a random sample of frames (annotated + noise-only)

Visually inspect a reproducible random sample before committing to the full sweep: a few
**annotated** frames (GT regions present) and a **noise-only** frame (no annotations), each rendered
as `[GT | coherent_power | cuda_dino]`. Change `seed` for a different draw, or `n_annotated`/`n_noise`
for more frames.

In [ ]:
# Reproducible random review: N annotated frames + M noise-only frames, each as a 3-panel.
# Frames are classified annotated vs noise-only by GT annotation count (shared across detectors).
from IPython.display import display

SAMPLE = v.sample_review_frames(BATCH_ROOT, FILE_STEM, n_annotated=5, n_noise=1, seed=7)
print(f"{SAMPLE['annotated_available']} annotated / {SAMPLE['noise_available']} noise-only frames available")
print("reviewing frames:", SAMPLE['review_frames'], " (noise-only:", SAMPLE['noise_frames'], ")")

for fr in SAMPLE['review_frames']:
    b = v.load_frame_bundle_smart(BATCH_ROOT, fr, file_stem=FILE_STEM,
                                  detectors=DETECTORS, capture_dirs=CAPTURE_DIRS)
    tag = 'NOISE-ONLY' if fr in SAMPLE['noise_frames'] else f'{len(b.gt_items)} GT regions'
    print(f"--- frame {fr}: {tag} | detectors {list(b.detector_masks)} ---")
    display(v.plot_frame_panels(b))

## (optional) Per-frame metrics for context

Pixel precision/recall/F1/IoU and per-region coverage for the same frame.
Reminder: GT masks are *filled bounding boxes*, so sparse detectors show high precision /
low recall; lean on region coverage + a tuned coverage threshold for 'did it find the signal'.

In [ ]:
import math, statistics
src_meta = None
for d in CAPTURE_DIRS:
    cand = Path(d) / f'{file_stem}.sigmf-meta'
    if cand.exists():
        src_meta = cand; break
for det in (DETECTORS or detectors_present):
    run_dir = layout['batch_root'] / det / file_stem
    fr, rr = mem.evaluate_run(run_dir, det, file_stem, sigmf_meta_path=src_meta, frame_limit=FRAME)
    row = next((r for r in fr if r['frame_number'] == FRAME), None)
    if row:
        flag = '' if row.get('mask_present', True) else '  [NO MASK emitted for this frame]'
        print(f"{det}: P={row['precision']:.3f} R={row['recall']:.3f} "
              f"F1={row['f1']:.3f} IoU={row['iou']:.3f} FParea={row['fp_area_fraction']:.4f}{flag}")

## (optional) Aggregate detection rate by breakdown (needs pandas)

Run `eval_detector_masks.py --batch-root ... --out-dir batch_runs/<run_id>` first to build
the tidy tables, then group at read time. Adding a breakdown dimension is one entry in
`mask_eval_metrics.BUCKETERS`.

In [ ]:
# import pandas as pd
# region_df = pd.read_parquet('batch_runs/<run_id>/region_metrics.parquet')
# mem.detection_rate_by(region_df, 'signal_class')
# mem.detection_rate_by(region_df, 'bandwidth')
# mem.detection_rate_by(region_df, 'attenuation_db')